In [ ]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [ ]:
def gen_pseudo_data(rows=10):
    prompt = f"""
    Generate {rows} server status records.
    Fields: hostname[a-z0-9.]{12,20}, ip address(v4,v6), disk usage (%), cpu utilization (%), last updated (YYYY-MM-DD HH:mm) 2023-2025.
    Condition: To look like real data, rather than continuous or similar forms of data, vary the values across records.
    Return the data strictly in JSON format as a list of objects for transformation into a DataFrame.
    """
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that outputs only JSON."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    raw_data = json.loads(response.choices[0].message.content)
    data_list = list(raw_data.values())[0] if isinstance(raw_data, dict) else raw_data
    return pd.DataFrame(data_list)

df = gen_pseudo_data()
df.head()

,hostname,ip_address,disk_usage,cpu_utilization,last_updated
0,server1234.example.com,192.168.1.45,72,55,2024-01-15 14:23
1,host5678.example.com,2001:0db8:85a3:0000:0000:8a2e:0370:7334,35,30,2024-02-20 09:12
2,srv91011.example.com,172.16.254.1,88,78,2025-03-05 17:45
3,example12.test.com,10.0.0.54,45,24,2023-10-30 22:00
4,data14.example.com,192.168.100.55,92,85,2024-04-12 08:15


In [ ]:
#Excelの課題完了

#df.to_excel("task_01.src.xlsx", index=False)
df = pd.read_excel('task_01.src.xlsx')
df.head()

def add_uptime(input_date):
    prompt = f"""
    Return duration since {input_date}[YYYY-MM-DD HH:mm] to current epoch time.
    Rules:
    1. Only output the duration in format 'X days Y hours'. 
    2. No explanations, no markdown, no backticks.
    Expected result: =DATEDIF(NOW(), "{input_date}", "D") & " days " & HOUR(NOW()-("{input_date}")) & " hours"
    """
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are an Excel formula generator. You never talk, you only output formulas."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=50
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {e}"

def check_replace_candidate(row):
    disk = row['disk_usage']
    cpu = row['cpu_utilization']
    prompt = f"""
    CPU utilization is {cpu}% and Disk usage is {disk}%. 
    If CPU >= 50 AND Disk >= 75, output 'Y'. Otherwise, output 'N'.
    Output ONLY 'Y' or 'N' without any explanation or markdown.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are a logical classifier. Answer only with Y or N."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return "N"


df['Uptime'] = df['last_updated'].apply(add_uptime)
df['replace_candidates'] = df.apply(check_replace_candidate, axis=1)

df.to_excel("task_01_result.xlsx", index=False)



In [33]:
df.head()

,hostname,ip_address,disk_usage,cpu_utilization,last_updated,Uptime,replace_candidates
0,server1234.example.com,192.168.1.45,72,55,2024-01-15 14:23,"=DATEDIF(""2024-01-15 14:23"", NOW(), ""D"") & "" d...",N
1,host5678.example.com,2001:0db8:85a3:0000:0000:8a2e:0370:7334,35,30,2024-02-20 09:12,"=DATEDIF(""2024-02-20 09:12"", NOW(), ""D"") & "" d...",N
2,srv91011.example.com,172.16.254.1,88,78,2025-03-05 17:45,"=DATEDIF(""2025-03-05 17:45"", NOW(), ""D"") & "" d...",Y
3,example12.test.com,10.0.0.54,45,24,2023-10-30 22:00,"=DATEDIF(""2023-10-30 22:00"", NOW(), ""D"") & "" d...",N
4,data14.example.com,192.168.100.55,92,85,2024-04-12 08:15,"=DATEDIF(""2024-04-12 08:15"", NOW(), ""D"") & "" d...",Y
